# Baseline đối sánh: trung bình cùng thứ trong bốn tuần

Đây là mốc tham khảo để nhóm so sánh các phương pháp dự báo. Mỗi cặp cửa hàng-nhóm hàng có một giá trị dự báo cho từng thứ, bằng **trung bình số học sales của cùng thứ trong 28 ngày lịch trước và gồm mốc dự báo**. Giữ số 0 thật, không biến đổi log trước khi lấy trung bình.

Mặc định giữ lại 16 ngày cuối train để đánh giá. Dự báo toàn bộ khoảng này từ một mốc duy nhất; lịch sử không được cập nhật bằng sales thực của validation. TV4 có thể đổi mốc bằng tham số, sau đó các thành viên sử dụng cùng mốc, ID và hàm RMSLE để đối chiếu.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "store_sales").is_dir() and (p / "data" / "raw").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from store_sales.baseline import run_baseline
from store_sales.metrics import evaluate_by_id


## Chạy và lưu kết quả thật

Lệnh tương đương: `python scripts/run_baseline.py`. Với dữ liệu hiện tại, lịch sử dùng 03/07-30/07/2017, validation là 31/07-15/08/2017. Ngày trong tương lai chỉ dùng để xác định thứ; sales thực được mở ra sau khi đã dự báo để tính lỗi.

Khi không có quan sát cùng thứ trong cửa sổ, dùng trung bình gần đây của chính chuỗi đó. Nếu chuỗi hoàn toàn không có lịch sử trong cửa sổ thì dự báo 0; số lần dùng từng quy tắc được ghi trong kết quả.


In [2]:
result = run_baseline(
    ROOT / "data" / "raw",
    ROOT / "outputs" / "baseline",
    ROOT / "reports",
)
display(pd.Series({key: result[key] for key in [
    "method", "aggregation", "origin", "history_start", "history_end",
    "validation_start", "validation_end", "n_rows", "n_series", "rmsle"
]}, name="Baseline validation").to_frame())
display(pd.Series(result["fallback_counts"], name="Số dòng theo quy tắc dự báo").to_frame())


,Baseline validation
method,same_weekday_mean_28d
aggregation,arithmetic_mean
origin,2017-07-30
history_start,2017-07-03
history_end,2017-07-30
validation_start,2017-07-31
validation_end,2017-08-15
n_rows,28512
n_series,1782
rmsle,0.53113


,Số dòng theo quy tắc dự báo
same_weekday,28512


## Đối chiếu lại bằng ID

CSV bên dưới là dự báo **validation**, có nhãn thật để kiểm tra lỗi; không phải file nộp Kaggle. Hàm đánh giá ghép theo ID, không phụ thuộc thứ tự các dòng mà mô hình khác trả về.


In [3]:
predictions = pd.read_csv(ROOT / "outputs" / "baseline" / "validation_predictions.csv")
actual = predictions[["id", "sales_actual"]].rename(columns={"sales_actual": "sales"})
forecast = predictions[["id", "sales_pred"]].sample(frac=1, random_state=42)
checked = evaluate_by_id(actual, forecast)
assert checked["n_rows"] == result["n_rows"]
assert abs(checked["rmsle"] - result["rmsle"]) < 1e-12
print("Ghép lại theo ID sau khi đảo thứ tự dự báo: khớp điểm baseline.")
display(Markdown((ROOT / "reports" / "baseline_report.md").read_text(encoding="utf-8")))


Ghép lại theo ID sau khi đảo thứ tự dự báo: khớp điểm baseline.


# Baseline tham khảo - trung bình cùng thứ

Baseline là quy tắc dự báo đơn giản để đối chiếu, không lựa chọn mô hình ML cho các thành viên.

- Mốc cuối được biết sales: **2017-07-30**.
- Cửa sổ lịch sử: **2017-07-03 đến 2017-07-30**, 28 ngày lịch.
- Validation: **2017-07-31 đến 2017-08-15**, 16 ngày.
- Chấm **28,512 dòng**, 1,782 chuỗi; giữ nguyên số 0 và lượng bán thập phân.

## Cách dự báo

Với mỗi cửa hàng-nhóm hàng-ngày đích, lấy trung bình cộng sales của các ngày cùng thứ trong cửa sổ lịch sử. Toàn bộ khoảng dự báo dùng cùng một mốc lịch sử; không cập nhật bằng sales thật của validation.
Nếu không có ngày cùng thứ, dùng trung bình cửa hàng-nhóm hàng trong cửa sổ; nếu không có lịch sử của chuỗi thì dùng 0. Cả ba trường hợp đều được đếm trong JSON.

## Kết quả thực tế

| Phương pháp | RMSLE validation |
|---|---:|
| Trung bình cùng thứ, 28 ngày lịch sử | **0.531130** |

RMSLE = sqrt(mean((log1p(dự báo) - log1p(thực tế))²)); càng thấp càng tốt. Tính trên toàn bộ dòng sau khi đối chiếu ID, không lấy trung bình điểm từng nhóm hàng để thay thế.

Nguồn dự báo: same_weekday: 28,512.

## Bàn giao và giới hạn

- Dự báo đối chiếu và nhãn validation: `outputs/baseline/validation_predictions.csv`. Đây là tệp nội bộ, không phải submission Kaggle.
- `baseline_metrics.json` ghi mốc, cửa sổ, điểm, số dòng, checksum nguồn/dự báo và môi trường.
- Các phương pháp khác cần dùng cùng origin, các ID/ngày validation và hàm RMSLE để so sánh công bằng. TV4 có thể thống nhất thêm các mốc khác với nhóm.
- Đây là một holdout tham khảo; chưa kết luận chất lượng trên các giai đoạn khác hoặc điểm Kaggle. Baseline chưa dùng khuyến mãi, ngày lễ hay xu hướng.
- Sales được giữ ở thang gốc; log1p chỉ xuất hiện trong phép chấm RMSLE. Không có scaler được fit hoặc mô hình ML được huấn luyện.
- CSV gốc được đối chiếu checksum trước/sau và không thay đổi.


## Dùng kết quả để so sánh

- Các phương pháp dùng cùng tập dòng validation, cùng khoảng 16 ngày và cùng mốc cuối dữ liệu quan sát.
- Gọi `evaluate_by_id(actual, predicted)` với các cột `id,sales` cho nhãn và `id,sales_pred` cho dự báo. Không âm, không NaN/vô hạn, không thiếu hoặc trùng ID.
- RMSLE càng thấp càng tốt trên cùng phép kiểm định. Đây là điểm validation nội bộ, không phải điểm Kaggle hoặc bảo đảm cho mọi giai đoạn tương lai.
- Không dùng điểm này để kết luận phương pháp nào tốt hơn khi chưa có kết quả thật của các phương pháp đó. Baseline bỏ qua khuyến mãi/ngày lễ; các thành viên vẫn tự chọn mô hình.
- Khi thay mốc hoặc định nghĩa baseline, cần chạy và ghi lại điểm tham chiếu tương ứng. Báo cáo hiện chỉ đánh giá một khoảng, không tuyên bố ổn định trên nhiều giai đoạn.
